# 4 · Tonhöhenraster und Export

[![In Colab öffnen](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sem-16161d/GreasyPidginTutorial/blob/main/DE/jupyter/markov-book-de/04_tonhoehe_und_export.ipynb)

Bisher konnte die Kette nur Töne erzeugen, die sie gesehen hat. Jetzt formen wir das **Tonmaterial** mit dem
`PitchGrid` von GreasyPidgin und exportieren das Ergebnis als MIDI, Notenbild (MusicXML), CSV und JSON.

In [ ]:
# Einrichtung -- läuft von selbst (beim ersten Mal etwa 20 Sekunden)
import importlib.util, pathlib, subprocess, sys, urllib.request

GP_REF        = "main"   # welche Version von GreasyPidgin verwendet wird (Git-Branch oder -Tag)
TUTORIAL_REF  = "main"   # welche Version der Hilfsdatei dieses Tutorials verwendet wird
GP_REPO       = "sem-16161d/GreasyPidgin"
TUTORIAL_REPO = "sem-16161d/GreasyPidginTutorial"

if importlib.util.find_spec("GreasyPidgin") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check",
                    f"git+https://github.com/{GP_REPO}.git@{GP_REF}"], check=True)

if not pathlib.Path("workshop_helpers.py").exists():
    urllib.request.urlretrieve(
        f"https://raw.githubusercontent.com/{TUTORIAL_REPO}/{TUTORIAL_REF}/DE/jupyter/markov-book-de/workshop_helpers.py",
        "workshop_helpers.py")

import random
import matplotlib.pyplot as plt
from GreasyPidgin import MarkovChain, MarkovRule, MarkovRules
import workshop_helpers as wh

wh.set_language("de")    # deutsche Meldungen, Beschriftungen und Notennamen
print("Bereit!")

In [ ]:
from GreasyPidgin import PitchGrid

midi_path = wh.choose_midi()
melody = wh.load_melody(midi_path)

chain = MarkovChain()
chain.generateRulesFromList(melody.pitches, order=2)
new_pitches = chain.getNElements(seed=tuple(melody.pitches[:2]), n=32)

wh.render_midi(new_pitches, melody.deltas, melody.bpm, "original_scale.mid")
print("Eine Melodie aus deiner Kette, wie in Kapitel 2:")
wh.listen("original_scale.mid")

## In welcher Tonleiter steht sie?

Ein `PitchGrid` ist die Menge der erlaubten Tonhöhen. `fromCorrelation` vergleicht die Noten deiner Melodie
mit einer Liste von Tonleitern und gibt die beste Übereinstimmung als Raster zurück. Wir lassen es unter einigen
gängigen Tonleitern und allen zwölf möglichen Grundtönen wählen.

In [ ]:
scale = PitchGrid.fromCorrelation(
    melody.pitches, 36, 96,
    possibleSystems    = ["edo12"],
    possibleMasks      = ["ionian", "aeolian", "dorian", "mixolydian", "majorPentatonic", "minorPentatonic"],
    possibleScaleRoots = list(range(60, 72)),
)
print("beste Übereinstimmung:", wh.pitch_class_name(scale.scaleRoot), scale.maskName)
print("Tonhöhen:", [wh.pitch_name(p) for p in sorted(scale)[:14]], "...")

## Die Melodie in eine andere Tonleiter schieben

`quantise` verschiebt jede Note auf die nächste Tonhöhe des Rasters. Hier: c-Moll (äolisch) —
dieselbe Melodie mit nach unten gezogener Terz, Sexte und Septime.

In [ ]:
other = PitchGrid.fromMask("edo12", "aeolian", 36, 96, "c4", "c4")

in_minor = [other.quantise(float(p)) for p in new_pitches]
print(" ".join(wh.pitch_name(p) for p in in_minor))

wh.render_midi(in_minor, melody.deltas, melody.bpm, "in_minor.mid")
wh.listen("in_minor.mid")

**Probier es aus:** Ersetze `"aeolian"` durch `"dorian"`, `"phrygian"`, `"lydian"`, `"wholeTone"`,
`"majorPentatonic"` oder `"harmonicMinor"`.

## Jenseits des Klaviers: andere Stimmungen

GreasyPidgin kennt viele Stimmungssysteme. In einer historischen Stimmung wie Werckmeister I liegen die Stufen der
Tonleiter nicht genau auf den Klaviertasten, die Tonhöhen sind also **gebrochene MIDI-Nummern**.
Beim Export werden daraus MIDI-*Pitch-Wheel*-Nachrichten (und der Exporter schreibt die Abweichung in Cent
als Textmarke dazu).

In [ ]:
werck = PitchGrid.fromMask("werckmeister1", "ionian", 36, 96, "c4", "c4")
detuned = [werck.quantise(float(p)) for p in new_pitches]

print([round(p, 2) for p in detuned[:10]], "...")

## Export

Alles bisher wurde als **Gesture** gebaut. Eine Gesture kann sich in mehreren Formaten selbst schreiben.
Wir nehmen die Moll-Version:

In [ ]:
gesture = wh.render_midi(in_minor, melody.deltas, melody.bpm, "melody.mid")

gesture.toCsv("melody.csv")      # eine Tabelle: eine Zeile pro Note
gesture.toJson("melody.json")    # alle Details als strukturierte Daten

In [ ]:
# MusicXML braucht die Bibliothek music21 (wird hier nur installiert, falls sie fehlt)
import importlib.util, subprocess, sys
if importlib.util.find_spec("music21") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check", "music21"], check=True)

gesture.toXml("melody.xml", title="Eine Markov-Melodie")

| Datei | wofür |
|---|---|
| `melody.mid` | jede DAW, jedes Notationsprogramm, jeder Synthesizer |
| `melody.xml` | MuseScore, Dorico, Finale, Sibelius: öffnet sich als Notenbild |
| `melody.csv` | Excel / pandas: die Zahlen ansehen (oder bearbeiten) |
| `melody.json` | andere Programme und Skripte |

Alle vier Dateien in einem Download:

In [ ]:
wh.download_zip(["melody.mid", "melody.xml", "melody.csv", "melody.json"], "my_markov_melody.zip")

## Wie geht es weiter?

- Kombiniere deine Ketten mit den **Hüllkurven** von GreasyPidgin für Dynamik, **L-Systemen** für
  Formverläufe oder **zellulären Automaten** für Rhythmus.
- Lerne Ketten aus *mehreren* MIDI-Dateien gleichzeitig, indem du ihre Tonhöhenlisten verbindest, bevor du
  `generateRulesFromList` aufrufst.
- Nimm die Ausgabe einer Kette als Eingabe der nächsten.